In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
df = pd.read_csv("produk_tokopedia.csv")
print(df.head())
print(f"\nShape: {df.shape}")
print(f"\nMissing Values:\n{df.isnull().sum()}")


                                         Nama Produk        Nama Toko  \
0  Raw Food Beef Bone-In / Daging Giling / Dog Fo...  Lazy Dog Supply   
1  Turkey Meat ONLY NO BONE - 1KG - Daging Kalkun...       RumahBully   
2  Chicken mix Salmon Raw Food ayam mix salmon ma...       RumahBully   
3  PROMO BULANAN - SOSIS CHOP PREMIUM RAW - DOG F...       RumahBully   
4  Premium Dog Food / Daging Sapi Giling / Raw Fo...       RumahBully   

         Lokasi Toko       Terjual Jumlah Ulasan  Rating  Harga (IDR)  \
0    Jakarta Selatan  1rb+ terjual     10 ulasan     5.0        35000   
1      Jakarta Timur  250+ terjual           NaN     5.0        79999   
2      Jakarta Barat    26 terjual           NaN     5.0        33000   
3  Tangerang Selatan  100+ terjual           NaN     5.0        29900   
4    Jakarta Selatan  1rb+ terjual           NaN     5.0        39000   

   Diskon (%)                                         Produk URL  
0         0.0  https://www.tokopedia.com/lazy-dog-suppl

In [ ]:
# Prepocessing untuk kolom "Terjual"
def prepocessing_terjual(sentence):
    if pd.isna(sentence): #Handling missing values (jika teks kosong)
        return 0 
    
    #1. Case Folding
    text = str(sentence).lower() 

    #2. Stopword removal [Membuang karakter yang tidak memiliki nilai matematis]
    custom_stopwords = ['terjual', '+', ' ']
    for stopword in custom_stopwords:
        text = text.replace(stopword, '')

    # Normalization + Feature Extraction
    if 'rb' in text:
        text = text.replace('rb', '')
        try:
            return int(float(text) * 1000) #ganti 'rb' jdi int
        except ValueError:
            return 0

    
    elif 'jt' in text:
        text = text.replace('jt', '') #Handle satuan 'jt'
        try:
            return int(float(text) * 1_000_000)
        except ValueError:
            return 0
        
    else:
        try:
            return int(float(text))
        except ValueError:
            return 0
# Tes
df['Terjual_bersih'] = df['Terjual'].apply(prepocessing_terjual)
print(df[['Terjual', 'Terjual_bersih']].head(10))

        Terjual  Terjual_bersih
0  1rb+ terjual            1000
1  250+ terjual             250
2    26 terjual              26
3  100+ terjual             100
4  1rb+ terjual            1000
5  100+ terjual             100
6  4rb+ terjual            4000
7  1rb+ terjual            1000
8  500+ terjual             500
9   30+ Terjual              30


In [ ]:
# Prepocessing untuk kolom "Jumlah Ulasan"
def prepocessing_ulasan(sentence):
    if pd.isna(sentence):
        return 0
    
    text = str(sentence).lower().strip()

    # Stopword removal
    custom_stopwords = ['ulasan', '+', ' ']
    for stopword in custom_stopwords:
        text = text.replace(stopword, '')

    #  Feature Extraction
    try:
        return int(float(text))
    except ValueError:
        return 0
    
# Terapkan fungsi ke kolom Jumlah Ulasan
df['Ulasan_bersih'] = df['Jumlah Ulasan'].apply(prepocessing_ulasan)

# Tes
print(df[['Jumlah Ulasan', 'Ulasan_bersih']].head(10))

  Jumlah Ulasan  Ulasan_bersih
0     10 ulasan             10
1           NaN              0
2           NaN              0
3           NaN              0
4           NaN              0
5           NaN              0
6           NaN              0
7           NaN              0
8           NaN              0
9     29 ulasan             29


In [ ]:
# Fitur tambahan

# Harga setelah diskon
df['Harga_setelah_diskon'] = df['Harga (IDR)'] * (1 - df['Diskon (%)']/100)

# Apakah ada diskon? (0 atau 1)
df['Ada_diskon'] = (df['Diskon (%)'] > 0).astype(int)

# Interaksi Rating x Ulasan, karena rating 5.0 dari 1 ulasan sangat berbeda dengan rating 4.8 dari 500 ulasan
df['Skor_kepercayaan'] = df['Rating'] * df['Ulasan_bersih']

print(df[['Harga (IDR)', 'Diskon (%)', 'Harga_setelah_diskon', 'Ada_diskon', 'Skor_kepercayaan']].head())

   Harga (IDR)  Diskon (%)  Harga_setelah_diskon  Ada_diskon  Skor_kepercayaan
0        35000         0.0               35000.0           0              50.0
1        79999         0.0               79999.0           0               0.0
2        33000         0.0               33000.0           0               0.0
3        29900         0.0               29900.0           0               0.0
4        39000         0.0               39000.0           0               0.0


In [ ]:
# Hapus outllier pada Harga
Q1 = df['Harga (IDR)'].quantile(0.01) # 1% Terbawah 
Q3 = df['Harga (IDR)'].quantile(0.99) # 1% Teratas
df = df[(df['Harga (IDR)'] >= Q1) & (df['Harga (IDR)'] <= Q3)]

print(f"Shape: {df.shape}")
print(f"Harga min: Rp {df['Harga (IDR)'].min():,}")
print(f"Harga max: Rp {df['Harga (IDR)'].max():,}")

Shape: (28929, 14)
Harga min: Rp 2,019
Harga max: Rp 13,200,000


In [ ]:
# Membuat label target (Y)
THRESHOLD = df['Terjual_bersih'].quantile(0.75) # Kita pakai batas 250 pcs (Top 25% produk paling laku)
print(f"Threshold (persentil 75): {THRESHOLD} pcs")

# Bikin kolom baru, jika terjual > 250 maka 1, jika tidak maka 0
df['is_bestseller'] = (df['Terjual_bersih'] > THRESHOLD).astype(int)

print("Distribusi Target (1 = Best Seller, 0 = Biasa):")
print(f"Rasio Best Seller {df['is_bestseller'].mean()*100:.1f}")


Threshold (persentil 75): 250.0 pcs
Distribusi Target (1 = Best Seller, 0 = Biasa):
Rasio Best Seller 23.4


In [ ]:
FEATURES = [
    'Harga (IDR)',
    'Diskon (%)',
    'Rating',
    'Ulasan_bersih',
    'Harga_setelah_diskon',
    'Ada_diskon',
    'Skor_kepercayaan',

]
X = df[FEATURES]
Y = df['is_bestseller']

print(f"\nShape X: {X.shape}")
print(f"Shape Y: {Y.shape}")
print(f"Fitur2 yg dipakai: {FEATURES}")


Shape X: (28929, 7)
Shape Y: (28929,)
Fitur2 yg dipakai: ['Harga (IDR)', 'Diskon (%)', 'Rating', 'Ulasan_bersih', 'Harga_setelah_diskon', 'Ada_diskon', 'Skor_kepercayaan']


In [ ]:
# Splitting data Train & Test
from sklearn.model_selection import train_test_split

# 80% Train & 20% Test
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, 
    test_size=0.2, 
    random_state=42,
    stratify=Y
    )

print("\nJumlah data: Train", len(X_train)," Test: ", len(X_test))
print(f"Proporsi best seller train: {Y_train.mean()*100:.1f}%")
print(f"Proporsi best seller test:  {Y_test.mean()*100:.1f}%")


Jumlah data: Train 23143  Test:  5786
Proporsi best seller train: 23.4%
Proporsi best seller test:  23.3%


In [ ]:
# Model Training memakai RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

model_rf = RandomForestClassifier(
    n_estimators=200, 
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
    )

# Training
print("Training Random Forest...")
model_rf.fit(X_train, Y_train)
print("Training Done!\n")

# Cross-Validation (CV)
# Kita bagi data train menjadi 5 bagian, train 4 bagian, test 1 bagian,
# lakukan 5 kali (rotasi). Hasilnya lebih reliable daripada test sekali saja.
# Kalau akurasi CV jauh lebih rendah dari akurasi test biasa → model overfitting!
cv_scores = cross_val_score(model_rf, X_train, Y_train, cv=5, scoring='roc_auc', n_jobs=1)
print(f"ROC-AUC tiap fold: {[f'{s:.3f}' for s in cv_scores]}")
print(f"Rata-rata: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Evaluasi dengan ROC-AUC, bukan hanya accuracy (Score 1.0 = sempurna, 0.5 = sama dengan tebak acak)
importances = pd.Series(model_rf.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=False)

for feat, imp in importances.items():
    bar = "█" * int(imp * 50)
    print(f"{feat:<25} {imp:.4f}  {bar}")


Training Random Forest...
Training Done!

ROC-AUC tiap fold: ['0.885', '0.899', '0.877', '0.898', '0.887']
Rata-rata: 0.889 ± 0.008
Rating                    0.1896  █████████
Harga_setelah_diskon      0.1863  █████████
Skor_kepercayaan          0.1817  █████████
Ulasan_bersih             0.1790  ████████
Harga (IDR)               0.1744  ████████
Diskon (%)                0.0714  ███
Ada_diskon                0.0176  


In [ ]:

IMPORTANCES = dict(zip(FEATURES, model_rf.feature_importances_))


STATS = {
    'total_produk': len(df),
    'bestseller_rate': df['is_bestseller'].mean(),
    'median_harga': df['Harga (IDR)'].median(),
    'median_diskon': df['Diskon (%)'].median(),
    'median_rating': df['Rating'].median(),
    'median_ulasan': df['Ulasan_bersih'].median(),
    'pct25_harga': df['Harga (IDR)'].quantile(0.25),
    'pct75_harga': df['Harga (IDR)'].quantile(0.75),
    'pct25_terjual': df['Terjual_bersih'].quantile(0.25),
    'pct75_terjual': df['Terjual_bersih'].quantile(0.75),
    'pct90_terjual': df['Terjual_bersih'].quantile(0.90),
}


STATS['feature_importances'] = IMPORTANCES



model_package = {
    'model': model_rf,        
    'features': FEATURES,
    'threshold': THRESHOLD,
    'market_stats': STATS      
}
import pickle
with open("model_bestseller.pickle", "wb") as file:
    pickle.dump(model_package, file)

print("\n✅ 'model_bestseller.pickle' berhasil disimpan dengan statistik lengkap!")
print(f"Fitur yang disimpan: {FEATURES}")





'model_bestseller.pickle' berhasil disimpan!
Fitur yang disimpan: ['Harga (IDR)', 'Diskon (%)', 'Rating', 'Ulasan_bersih', 'Harga_setelah_diskon', 'Ada_diskon', 'Skor_kepercayaan']
